Dedup in this stage, set data types including time normalization

In [0]:
from delta.tables import DeltaTable

silver_path = "/Volumes/ingestion/raw_ads/raw/delta"
checkpoint_silver_path = "/Volumes/ingestion/raw_ads/raw/checkpoints"

# Read the bronze stream
bronze_stream = (
    spark.readStream
        .format("delta")
#        .load("/Volumes/ingestion/raw_ads/raw/delta/ads_bronze")
        .load("/Volumes/ingestion/raw_ads/raw/delta/ads_bronze_file")
)

# data types to cast columns to
from pyspark.sql.functions import col, to_timestamp


In [0]:

# Cast timestamp and floats

silver_df = bronze_stream.select(
    col("event_id"),
    col("event_type"),
    to_timestamp("event_time").alias("event_time"),
    col("campaign_id"),
    col("user_id"),
    col("session_id"),
    col("geo"),
    col("device"),
    col("price_paid").cast("double"),
    col("conversion_value").cast("double")
)

# dedup events, up to 10 minutes late
silver_df = (
    silver_df
    .withWatermark("event_time", "10 minutes")
    .dropDuplicates(["event_id"])
)

# write silver table
silver_query = (
    silver_df.writeStream
        .format("delta")
        .outputMode("append")
        # use ads_silver for prod
        #.option("checkpointLocation", "/Volumes/ingestion/raw_ads/raw/checkpoints/stg_ads_event")
        #.start("/Volumes/ingestion/raw_ads/raw/delta/stg_ads_event")
        # use stg_ads_event for development
        .option("checkpointLocation", f"{checkpoint_silver_path}/stg_ads_event_file")
        .trigger(availableNow=True)
        .start(f"{silver_path}/stg_ads_event_file")
)



In [0]:
# Verify data in silver
try:
    file_silver_df = spark.read.format("delta").load(f"{silver_path}/stg_ads_event")
    print(f"Total records from silver processing: {file_silver_df.count()}")
    display(file_silver_df)
except Exception as e:
    if "PATH_NOT_FOUND" in str(e):
        print("⚠️ Delta table doesn't exist yet.")
        print("Upload JSON files to /Volumes/ingestion/raw_ads/raw/json_input/ and the stream will process them.")
    else:
        raise e


In [0]:
# streaming join of impressions and clicks for funnel analytics
impressions = bronze_stream.filter(col("event_type") == "ad_impression")
clicks = bronze_stream.filter(col("event_type") == "ad_click")

from pyspark.sql.functions import expr

joined = (
    impressions.alias("i")
    .join(
        clicks.alias("c"),
        expr("""
            i.session_id = c.session_id AND
            i.ad_id = c.ad_id AND
            c.event_time >= i.event_time
        """),
        "left"
    )
    .select(
        col("i.event_id").alias("impression_id"),
        col("i.campaign_id"),
        col("i.user_id"),
        col("i.session_id"),
        col("i.event_time").alias("impression_time"),
        col("c.event_time").alias("click_time")
    )
)

def upsert(batch_df, batch_id):

    delta_table = DeltaTable.forPath(spark, f"{silver_path}/stg_impression_click")

    (
        delta_table.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.impression_id = s.impression_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    joined.writeStream
        .foreachBatch(upsert)
        .option("checkpointLocation", f"{checkpoint_silver_path}/stg_impression_click")
        .trigger(availableNow=True)
        .start()
)

In [0]:
# Verify data in impressions+click funnel
# Verify data in silver
try:
    file_silver_df = spark.read.format("delta").load(f"{silver_path}/stg_impression_click")
    print(f"Total records from impresions+click join: {file_silver_df.count()}")
    display(file_silver_df)
except Exception as e:
    if "PATH_NOT_FOUND" in str(e):
        print("⚠️ Delta table doesn't exist yet.")
        print("Upload JSON files to /Volumes/ingestion/raw_ads/raw/json_input/ and the stream will process them.")
    else:
        raise e
